# Hodgkin–Huxley Modelling: Hands-On Workshop
University of Melbourne Neuroscience PhD Programme, 16 November 2026

This workshop introduces the NEURON simulation environment in Python.

Activities 1–3 use a single compartment to explore how ion channels generate action potentials.
Activities 4–5 use a multiple-compartment retinal ganglion cell model from Sheasby & Fohlmeister (1999) to explore how action potentials start and travel through a real neuron.

## Activities
1. **Single action potentials:** gating variables, conductances, and the threshold current
2. **Tonic firing:** long current pulses
3. **Altering conductances:** the effect of the Na⁺ conductance
4. **Action potential generation and propagation** in a multi-compartment neuron
5. **Axonal conduction:** conduction velocity and axon diameter

## Setup
Run these two cells first, in order. The first installs NEURON and compiles the ion-channel mechanisms; the second imports NEURON and the workshop's helper functions (in `sheasbyfohlmeister1999_utils.py`), so it must come after.

If anything misbehaves, use **Runtime → Restart session** and run again from the top.

In [ ]:
#@title ▶ Run this first: installs NEURON and fetches the workshop files (~1 min) { display-mode: "form" }
REPO_URL = "https://github.com/jseo977/UniMelb-PhD-Neuroscience-Computational-Modelling-Workshop.git"

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install NEURON and interactive-plot support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuron==9.0.2", "ipympl"], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()

    # Colab loads matplotlib before ipympl is installed, so register the new
    # 'widget' backend with matplotlib (otherwise %matplotlib widget fails)
    try:
        from matplotlib.backends import backend_registry
        new = [(n, m) for n, m in backend_registry._read_entry_points()
               if n.lower() not in backend_registry._backend_to_gui_framework]
        backend_registry._validate_and_store_entry_points(new)
    except Exception:
        pass   # plots will fall back to static images

    # Fetch the workshop files (.mod, morphology, utils) and move into that folder
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)

# Compile the .mod files (skipped if already compiled; delete the x86_64/arm64 folder to force a rebuild)
if not any(os.path.isdir(d) for d in ("x86_64", "arm64", "aarch64")):
    build = subprocess.run(["nrnivmodl"], capture_output=True, text=True)
    if build.returncode != 0:
        print(build.stdout[-3000:], build.stderr[-3000:])
        raise RuntimeError("Compiling the .mod files failed (see messages above)")

print("Setup complete. Working directory:", os.getcwd())

# Imports and settings used throughout the notebook (run after the setup cell)
import sys
if "google.colab" in sys.modules:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')   # interactive figures (zoom / pan / reset)
    except Exception:
        get_ipython().run_line_magic('matplotlib', 'inline')
        print("Interactive plots unavailable; using static plots.")
else:
    get_ipython().run_line_magic('matplotlib', 'inline')       # static figures (work in VS Code and Jupyter)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from neuron import h, gui

import importlib, sheasbyfohlmeister1999_utils
importlib.reload(sheasbyfohlmeister1999_utils)   # pick up changes to the utils file without restarting
from sheasbyfohlmeister1999_utils import (
    find_spikes, spike_half_width,                                     # spike detection
    make_figure, plot_hh_panels,                                       # plotting
    load_rgc_model, update_rx_3d, plot_final_rgc_with_electrode,       # full cell (Activities 4–5)
    dendrite_path_sites, plot_interactive_dashboard, record_segments, seg_xyz,
    spike_origin, furthest_in_view,
)

plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

PEAK_HEIGHT = -20   # mV : a peak above this counts as an action potential

## The single-compartment model
A single cylindrical compartment the same size as the soma of the retinal ganglion cell used in Activities 4–5, with only the three conductances of the Hodgkin–Huxley model: Na⁺, delayed-rectifier K⁺ and a leak. The leak is set so that the compartment's input resistance (~450 MΩ) is within the range recorded from retinal ganglion cells (~200–600 MΩ).

The cell below defines two functions and then builds the compartment:
- `build_soma_sc()` creates the section, sets its size and inserts the leak and the ion channels (`spike.mod`, with its other channels switched off).
- `run_soma_sc(...)` places a current clamp (`IClamp`) on it, sets up recordings with `h.Vector().record(...)`, initialises the model at rest and runs the simulation.

Run this cell once before Activities 1–3.

In [ ]:
def build_soma_sc():
    """A single cylindrical compartment the size of the soma of ctt3219b, with Hodgkin–Huxley
    Na and delayed-rectifier K channels and a leak. Other ion channels were removed for simplicity."""
    soma_sc = h.Section(name='soma_sc')
    soma_sc.L    = 27.69     # µm (gives the same membrane area as the soma, ~1216 µm²)
    soma_sc.diam = 13.98     # µm
    soma_sc.nseg = 1
    soma_sc.Ra   = 110       # ohm·cm
    soma_sc.cm   = 1         # µF/cm²

    # Leak. Chosen so that the compartment's input resistance (~450 MΩ) is within the range
    # recorded from retinal ganglion cells (~200–600 MΩ). The full model's leak (g_pas = 8e-6 S/cm²)
    # would give an isolated soma ~10 GΩ, because in the full cell the dendrites and axon carry
    # most of the current away.
    soma_sc.insert('pas')
    soma_sc.g_pas = 0.0002    # S/cm²
    soma_sc.e_pas = -62.5     # mV

    # Hodgkin–Huxley channels only (spike.mod): Na and delayed-rectifier K.
    # spike.mod also contains A-type K, Ca and Ca-activated K channels: switch them off.
    soma_sc.insert('spike')
    soma_sc.gnabar_spike = 0.080     # S/cm² : Na
    soma_sc.gkbar_spike  = 0.018     # S/cm² : delayed-rectifier K
    soma_sc.ena = 35.0               # mV
    soma_sc.ek  = -75.0              # mV
    return soma_sc


def run_soma_sc(soma_sc, amp_pa, dur, delay, tstop, dt):
    """Inject a current pulse (amp_pa in pA, dur in ms, starting at delay ms) into the compartment,
    run to tstop ms with time step dt ms, and return the recordings as arrays:
    t, v, m, h, n (gating variables) and i (injected current, pA)."""
    # 1. Stimulus: a current clamp in the middle of the compartment
    clamp = h.IClamp(soma_sc(0.5))             # 0.5 indicates middle of the compartment
    clamp.delay = delay                        # Sets the start of the square current injection
    clamp.dur   = dur                          # Sets the duration of the square current injection
    clamp.amp   = amp_pa / 1000                # IClamp works in nA

    # 2. Recordings: NEURON fills these vectors as the simulation runs
    rec = {'t': h.Vector().record(h._ref_t),
           'v': h.Vector().record(soma_sc(0.5)._ref_v),
           'm': h.Vector().record(soma_sc(0.5)._ref_m_spike),
           'h': h.Vector().record(soma_sc(0.5)._ref_h_spike),
           'n': h.Vector().record(soma_sc(0.5)._ref_n_spike),
           'i': h.Vector().record(clamp._ref_i)}

    # 3. Run: start at rest, then integrate to tstop
    h.dt      = dt                             # Time resolution of the simulation
    h.tstop   = tstop                          # The timepoint where the simulation stops
    h.finitialize(-62.5)                       # resting potential (mV); gates start at steady state

    h.fcurrent()
    h.frecord_init()

    h.continuerun(tstop)                       # 'continuerun'-ning this simulation until the defined 'tstop'

    out = {k: np.array(x) for k, x in rec.items()}
    out['i'] *= 1000                           # nA -> pA
    return out # Output the stimulus waveform. Simulation data is stored in the global 'soma_sc' variable


# Build the compartment. If you come back to Activities 1–3 after Activity 4, re-run this cell first.
soma_sc = build_soma_sc()
GNA_SC_DEFAULT = soma_sc.gnabar_spike     # S/cm² : default Na conductance (Activity 3)
GK_SC_DEFAULT  = soma_sc.gkbar_spike      # S/cm² : default delayed-rectifier K conductance (Activity 3)

## Activity 1: Single action potentials
### 1a. A brief current pulse
A brief pulse of current (`PULSE_AMP`, `PULSE_DUR` < 5 ms) is injected into the compartment. The figure shows, from top to bottom: the injected current, the membrane potential, the gating variables (m, h, n) and the Na⁺ and K⁺ conductances, all on the same time axis.

- g_Na = ḡ_Na · m³h and g_K = ḡ_K · n⁴ (delayed-rectifier K⁺).

**Questions:** how do the gating variables change over time? Do they change as you expect? Compare the conductance plot with the action potential: when does each conductance rise and fall relative to the upstroke and the downstroke?

In [ ]:
# Activity 1a: a brief current pulse

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (< 5 ms) (Fill this in yourself!)
PULSE_DELAY = 10     # ms
T_STOP      = 40     # ms
DT          = 0.01   # ms : time step
#############################

soma_sc = build_soma_sc()
GNA_SC_DEFAULT = soma_sc.gnabar_spike     # S/cm² : default Na conductance (Activity 3)
GK_SC_DEFAULT  = soma_sc.gkbar_spike      # S/cm² : default delayed-rectifier K conductance (Activity 3)

r = run_soma_sc(soma_sc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP, DT)
g_na = soma_sc.gnabar_spike * r['m']**3 * r['h'] * 1000    # mS/cm²
g_k  = soma_sc.gkbar_spike  * r['n']**4 * 1000             # mS/cm²

# Plot the results
fig, axes = make_figure(4, 1, figsize=(8, 8), sharex=True, gridspec_kw={'height_ratios': [1, 2, 2, 2]})
axes[0].plot(r['t'], r['i'], color='k')
axes[0].set_ylabel('IClamp (pA)')
axes[1].plot(r['t'], r['v'], color='k')
axes[1].set_ylabel('Vm (mV)')
axes[2].plot(r['t'], r['m'], label='m (Na act.)', color='#2a78d6')
axes[2].plot(r['t'], r['h'], label='h (Na inact.)', color='#eb6834')
axes[2].plot(r['t'], r['n'], label='n (K act.)', color='#1baf7a')
axes[2].set_ylabel('Gate')
axes[2].legend(loc='upper right')
axes[3].plot(r['t'], g_na, label='gNa', color='#2a78d6')
axes[3].plot(r['t'], g_k,  label='gK (delayed rect.)', color='#1baf7a')
axes[3].set_ylabel('g (mS/cm²)')
axes[3].set_xlabel('Time (ms)')
axes[3].legend(loc='upper right')
plt.tight_layout()
plt.show()

### 1b. Finding the threshold current
Choose a pulse duration (< 5 ms) and find the smallest current that fires an action potential. Start with `PULSE_AMP` below threshold, run the cell, and raise `PULSE_AMP` in steps (no smaller than 10 pA), re-running each time, until an action potential appears. The figure is the same as in 1a (from here on it is drawn by `plot_hh_panels`, in the utils file).

**Questions:** what is the threshold current for your pulse duration? Repeat for a shorter and a longer pulse: what happens to the threshold? Look at m and h just below and just above threshold: what is different?

In [ ]:
# Activity 1b: find the threshold current for your pulse duration

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself! Raise it step by step until the cell fires)
PULSE_DUR   = 0      # ms (< 5 ms) (Fill this in yourself!)
PULSE_DELAY = 10     # ms
T_STOP      = 40     # ms
DT          = 0.01   # ms : time step (keep it smaller than PULSE_DUR)
#############################

soma_sc = build_soma_sc()
r = run_soma_sc(soma_sc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP, DT)
fired = len(find_spikes(r['v'])) > 0
print(f"{PULSE_AMP} pA for {PULSE_DUR} ms: {'ACTION POTENTIAL' if fired else 'no action potential'}")

plot_hh_panels(r['t'], r['i'], [('Vm', r['v'], 'k')], r['m'], r['h'], r['n'],
               soma_sc.gnabar_spike, soma_sc.gkbar_spike,
               title=f"{PULSE_AMP} pA for {PULSE_DUR} ms")

## Activity 2: Tonic firing
A long current pulse (`PULSE_DUR` > 100 ms) is injected into the compartment. Run the cell for a range of currents, from weak to strong (e.g. 50, 100, 200, 300, 500, 800 and 1500 pA), and note the numbers printed below the figure each time.

**Questions:** what do you see as the current increases? How do the firing rate and the size and width of the action potentials change, and why? Follow the gating variables: what stops h from recovering between spikes at the highest currents? What happens at the highest currents (*depolarisation block*)? The "lowest Vm in the last 20 ms" tells them apart: during repetitive firing the membrane repolarises between spikes; in depolarisation block it stays depolarised.

In [ ]:
# Activity 2: a long current pulse

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (> 100 ms) (Fill this in yourself!)
PULSE_DELAY = 10     # ms
T_STOP      = 250    # ms
DT          = 0.01   # ms : time step
#############################

soma_sc = build_soma_sc()
r = run_soma_sc(soma_sc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP, DT)

plot_hh_panels(r['t'], r['i'], [('Vm', r['v'], 'k')], r['m'], r['h'], r['n'],
               soma_sc.gnabar_spike, soma_sc.gkbar_spike,
               title=f"{PULSE_AMP} pA for {PULSE_DUR} ms")

# Spikes during the pulse
pk = find_spikes(r['v'])
pk = pk[(r['t'][pk] >= PULSE_DELAY) & (r['t'][pk] <= PULSE_DELAY + PULSE_DUR)]
late = (r['t'] >= PULSE_DELAY + PULSE_DUR - 20) & (r['t'] <= PULSE_DELAY + PULSE_DUR)
print(f"Current:                     {PULSE_AMP} pA")
rate = len(pk) / (PULSE_DUR / 1000) if PULSE_DUR > 0 else 0     # Hz
print(f"Spikes during the pulse:     {len(pk)}  ({rate:.0f} Hz)")
if len(pk):
    print(f"Mean spike peak:             {r['v'][pk].mean():.1f} mV")
    print(f"Mean spike half-width:       {np.mean([spike_half_width(r['t'], r['v'], i) for i in pk]):.2f} ms")
print(f"Lowest Vm in the last 20 ms: {r['v'][late].min():.1f} mV")

## Activity 3: Altering conductances
`NA_X` multiplies the maximal Na⁺ conductance (`gnabar_spike`), and `K_X` multiplies the delayed-rectifier K⁺ conductance (`gkbar_spike`). Run the cell with `NA_X` = 0, 0.5, 1, 2, 6 and 10, first with a short pulse (as in Activity 1: e.g. 300 pA for 1 ms, `T_STOP` = 40 ms) and then with a long pulse (as in Activity 2: e.g. 300 pA for 200 ms, `T_STOP` = 250 ms). The defaults are restored at the end of the cell, so each run starts from the original model.

**Questions:** what changes as the Na⁺ conductance is reduced to zero, increased moderately, and increased a lot? Why does a very large Na⁺ conductance stop the cell firing during the long pulse? Why does the cell fire an action potential *before* the pulse starts when the Na⁺ conductance is very large, and why only one? Then try changing `K_X` instead.

In [ ]:
# Activity 3: scale the Na and K conductances

####### PARAMETERS ##########
NA_X        = 1      # multiplier on the Na conductance
K_X         = 1      # multiplier on the delayed-rectifier K conductance

PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (Fill this in yourself!)
PULSE_DELAY = 10     # ms
T_STOP      = 40     # ms
DT          = 0.01   # ms : time step
#############################

soma_sc.gnabar_spike = GNA_SC_DEFAULT * NA_X
soma_sc.gkbar_spike  = GK_SC_DEFAULT * K_X
r = run_soma_sc(soma_sc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP, DT)

plot_hh_panels(r['t'], r['i'], [('Vm', r['v'], 'k')], r['m'], r['h'], r['n'],
               soma_sc.gnabar_spike, soma_sc.gkbar_spike,
               title=f"gNa × {NA_X}, gK × {K_X}: {PULSE_AMP} pA for {PULSE_DUR} ms")

pk = find_spikes(r['v'])
before = pk[r['t'][pk] < PULSE_DELAY]
during = pk[(r['t'][pk] >= PULSE_DELAY) & (r['t'][pk] <= PULSE_DELAY + PULSE_DUR + 5)]
print(f"gNa = {soma_sc.gnabar_spike:.3f} S/cm²,  gK = {soma_sc.gkbar_spike:.3f} S/cm²")
print(f"Spikes before the pulse (spontaneous): {len(before)}")
print(f"Spikes in total:                       {len(pk)}")
if len(pk):
    print(f"Highest spike peak:                    {r['v'][pk].max():.1f} mV")

soma_sc = build_soma_sc() # restore the defaults
soma_sc.gnabar_spike = GNA_SC_DEFAULT
soma_sc.gkbar_spike  = GK_SC_DEFAULT

## Activity 4: Action potential generation and propagation
Now the full multi-compartment retinal ganglion cell of Sheasby & Fohlmeister (1999): a soma, a dendritic tree, an axon initial segment (AIS) and a 5.3 mm long axon.

### Set up the model
Load the morphology and insert the mechanisms. Run this cell once before Activities 4 and 5.

In [ ]:
# Activity 4 set-up: load the full retinal ganglion cell (run once before Activities 4 and 5)

####### PARAMETERS ##########
EL_X                    = 0       # µm : electrode position (0 = over the soma; 200 = over the axon)
EL_Y                    = 0       # µm
EL_HEIGHT               = 25      # µm : electrode height above the soma's surface
#############################

MORPH_DIR               = "neuron_morphology"   # folder holding the model's .asc and .hoc files
PLOT_ELECTRODE_POSITION = True    # True: show the cell and the electrode in 3D
PLOT_VIEW_UM            = 140     # µm : half-width of that 3D view

# Remove the single compartment (Activities 1–3) so it is not part of the full cell
if 'soma_sc' in globals():
    h.delete_section(sec=soma_sc)
    del soma_sc

load_rgc_model(MORPH_DIR)         # morphology, ion channels and extracellular mechanisms (see the utils file)

el_x, el_y, el_z = EL_X, EL_Y, EL_HEIGHT + h.soma.diam / 2 + 1   # +1 µm for the inner limiting membrane
update_rx_3d(el_x, el_y, el_z)
if PLOT_ELECTRODE_POSITION:
    plot_final_rgc_with_electrode(el_x, el_y, el_z, view_um=PLOT_VIEW_UM)

### Set up a brief pulse into the soma, and the recordings
A brief current pulse is injected into the soma, and Vm is recorded at the soma, the AIS, three points along a dendrite (blues) and three points along the axon (greens). The darker the colour, the further the site is from the soma. The gating variables are recorded at the soma.

**Finding the threshold:** as in Activity 1b, change `PULSE_AMP` here, then run this cell and the next one, until the cell fires. Then set `PULSE_AMP` at or just above threshold and continue.

**Question:** how does the threshold of the full cell compare with the single compartment in Activity 1b? Why might it be different?

In [ ]:
# Activity 4: a brief current pulse into the soma, recorded throughout the cell

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself! Find the threshold for your pulse duration)
PULSE_DUR   = 0      # ms (Fill this in yourself!)
PULSE_DELAY = 5      # ms
T_STOP      = 30     # ms (long enough for the spike to reach the far end of the axon)
DT          = 0.005  # ms : time step
#############################

# Stimulus: a current clamp in the middle of the soma
mc_iclamp = h.IClamp(h.soma(0.5))   # replaces the clamp from any earlier run of this cell
mc_iclamp.delay = PULSE_DELAY
mc_iclamp.dur   = PULSE_DUR
mc_iclamp.amp   = PULSE_AMP / 1000   # IClamp works in nA

# Recording sites along the longest dendrite and along the axon: (label, segment, µm from the soma)
dend_fractions = (0.25, 0.55, 0.9)   # fractions of the longest soma-to-tip dendritic path
axon_fractions = (0.1, 0.5, 0.9)     # positions along the 5.3 mm axon
dend_sites = [(f'Dendrite {d:.0f} µm', seg, d) for seg, d in dendrite_path_sites(fractions=dend_fractions)]
axon_sites = [(f'Axon {h.distance(h.soma(0.5), h.axon(x)) / 1000:.1f} mm', h.axon(x),
               h.distance(h.soma(0.5), h.axon(x))) for x in axon_fractions]
dend_dist = {label: d for label, seg, d in dend_sites}
axon_dist = {label: d for label, seg, d in axon_sites}

# Recording vectors define which compartments and variables to record
t_vec  = h.Vector().record(h._ref_t)                                                            # Time vector
i_vec  = h.Vector().record(mc_iclamp._ref_i)                                                    # Stimulus waveform
v_rec  = {name: h.Vector().record(sec(0.5)._ref_v)                                              # Voltage response at soma, initseg (AIS), and (distal) axon
          for name, sec in [('soma', h.soma), ('initseg', h.initseg), ('axon', h.axon)]}
v_dend = {label: h.Vector().record(seg._ref_v) for label, seg, d in dend_sites}                 # Voltage response at dendrite sites (25%, 55%, 90% of length away from soma)
v_axon = {label: h.Vector().record(seg._ref_v) for label, seg, d in axon_sites}                 # Voltage response at axon sites (10%, 50% and 90%of length away from soma)
m_soma = h.Vector().record(h.soma(0.5)._ref_m_spike)                                            # m gating variable (sodium activation)
h_soma = h.Vector().record(h.soma(0.5)._ref_h_spike)                                            # h gating variable (sodium inactivation)
n_soma = h.Vector().record(h.soma(0.5)._ref_n_spike)                                            # n gating variable (potassium activation)
v_all  = {sec.name(): h.Vector().record(sec(0.5)._ref_v) for sec in h.allsec()}                 # Every section, for the dashboard
v_segs = record_segments([h.soma, h.initseg, h.narrowr])                                        # Every segment of the soma, AIS and thin segment (zoomed-in dashboard)

# Styling
# Colours that encode distance from the soma. Dendrites = blue, Axon = green. Darker = Further
DEND_MAX_UM = 200                                  # longest dendritic path is ~190 µm
AXON_MAX_UM = h.distance(h.soma(0.5), h.axon(1))   # far end of the axon, ~5.5 mm
SITE_COLOURS = {'Soma': '#000000', 'AIS': '#e67e22'}
SITE_COLOURS.update({label: mcolors.to_hex(plt.cm.Blues(0.3 + 0.7 * d / DEND_MAX_UM)) for label, seg, d in dend_sites})
SITE_COLOURS.update({label: mcolors.to_hex(plt.cm.Greens(0.3 + 0.7 * d / AXON_MAX_UM)) for label, seg, d in axon_sites})

# Sites shown on the dashboard: (label, segment) and their Vm recordings
dash_sites = ([('Soma', h.soma(0.5)), ('AIS', h.initseg(0.5))]
              + [(label, seg) for label, seg, d in dend_sites + axon_sites])
dash_v = {'Soma': v_rec['soma'], 'AIS': v_rec['initseg'], **v_dend, **v_axon}

# Activity 4: run the simulation
h.dt      = DT
h.tstop   = T_STOP
h.celsius = 33                            # °C
h.finitialize(-62.5)                      # resting potential (mV)
for sec in h.allsec():                    # resting calcium in every section, so repeated runs agree
    if h.ismembrane('ca_ion', sec=sec):
        sec.cai = 0.0001
h.fcurrent()
h.frecord_init()
h.continuerun(T_STOP)

t  = np.array(t_vec)
v  = {name: np.array(vec) for name, vec in v_rec.items()}     # soma, AIS, axon
vd = {label: np.array(vec) for label, vec in v_dend.items()}  # dendrite sites
va = {label: np.array(vec) for label, vec in v_axon.items()}  # axon sites

peaks = find_spikes(v['soma'])
print(f"{PULSE_AMP} pA for {PULSE_DUR} ms: " +
      (f"ACTION POTENTIAL at the soma, t = {t[peaks[0]]:.2f} ms" if len(peaks) else "no action potential"))

grey = lambda: ('Soma', v['soma'], '#bbbbbb')       # soma, in grey, for reference
plot_hh_panels(t, np.array(i_vec) * 1000,
               [('Soma', v['soma'], SITE_COLOURS['Soma']), ('AIS', v['initseg'], SITE_COLOURS['AIS'])],
               m_soma, h_soma, n_soma, h.soma(0.5).gnabar_spike, h.soma(0.5).gkbar_spike,
               extra_panels=[('Dendrite Vm (mV)', [grey()] + [(l, tr, SITE_COLOURS[l]) for l, tr in vd.items()]),
                             ('Axon Vm (mV)',     [grey()] + [(l, tr, SITE_COLOURS[l]) for l, tr in va.items()])],
               title=f"{PULSE_AMP} pA for {PULSE_DUR} ms into the soma (gates and conductances at the soma)")

# Timing of the first action potential at the AIS vs the soma
ais_pk  = find_spikes(v['initseg'])
soma_pk = find_spikes(v['soma'])
if len(ais_pk) and len(soma_pk):
    t_ais, t_soma = t[ais_pk[0]], t[soma_pk[0]]
    print(f"AIS peak:  {t_ais:.3f} ms  ({v['initseg'][ais_pk[0]]:.1f} mV)")
    print(f"Soma peak: {t_soma:.3f} ms  ({v['soma'][soma_pk[0]]:.1f} mV)")
    print(f"Soma - AIS: {(t_soma - t_ais) * 1000:.0f} µs  "
          f"({'AIS first' if t_ais < t_soma else 'soma first' if t_soma < t_ais else 'simultaneous'})")
else:
    print(f"No action potential above {PEAK_HEIGHT} mV at the "
          f"{'AIS' if not len(ais_pk) else 'soma'}, so there is no peak to compare.")

# Spread through the cell: how far each site depolarises during the first somatic spike, and when it peaks
if len(soma_pk):
    t_sp   = t[soma_pk[0]]
    i_rest = int(np.searchsorted(t, PULSE_DELAY))                 # just before the current step
    soma_amp = v['soma'][soma_pk[0]] - v['soma'][i_rest]
    print(f"\n{'site':>18s}{'peak':>8s}{'depolarisation':>16s}{'% of soma':>11s}{'peak delay':>12s}")
    print(f"{'':>18s}{'(mV)':>8s}{'(mV)':>16s}{'':>11s}{'(ms)':>12s}")
    print(f"{'soma':>18s}{v['soma'][soma_pk[0]]:8.1f}{soma_amp:16.1f}{100:10.0f}%{0:12.2f}")
    for label, trace in list(vd.items()) + list(va.items()):
        pk_site = find_spikes(trace)
        pk_site = pk_site[t[pk_site] >= t_sp - 0.5]             # first spike at or after the somatic one
        if not len(pk_site):
            print(f"{label:>18s}   no spike reached this site before the end of the run")
            continue
        i_pk = pk_site[0]
        amp  = trace[i_pk] - trace[i_rest]
        print(f"{label:>18s}{trace[i_pk]:8.1f}{amp:16.1f}{100 * amp / soma_amp:10.0f}%{t[i_pk] - t_sp:12.2f}")

### Plot the results

### Interactive dashboard
The 3D view shows the membrane potential of every section; the traces show the soma, the AIS, the three dendrite sites (◆ on the cell, light to dark blue with distance) and the three axon sites (light to dark green; beyond the edge of the 3D view).

Press **▶ Play**, or drag the time slider. Click and drag the 3D view to rotate it, scroll to zoom, and hover over a ◆ to see its name.

In [ ]:
# Activity 4: interactive dashboard

####### PARAMETERS ##########
FRAME_MS     = 0.2   # ms of simulated time per animation frame
FRAME_DELAY  = 40    # ms of real time per frame (larger = slower playback)
COLOUR_GAMMA = 1     # 1 = linear colour scale; larger = more of the colours near the AP peak
#############################

dashboard = plot_interactive_dashboard(
    t, v_all,
    sites=[(label, seg, dash_v[label], SITE_COLOURS[label]) for label, seg in dash_sites],
    panels=[('Soma Vm',     ['Soma']),
            ('AIS Vm',      ['AIS']),
            ('Dendrite Vm', list(v_dend)),
            ('Axon Vm',     list(v_axon))],
    gates={'m': m_soma, 'h': h_soma, 'n': n_soma},
    protocol=[('IClamp', PULSE_DELAY, PULSE_DELAY + PULSE_DUR, 'orange')],
    frame_ms=FRAME_MS,
    frame_delay=FRAME_DELAY,
    colour_gamma=COLOUR_GAMMA,
    title=f"Somatic current injection: {PULSE_AMP} pA for {PULSE_DUR} ms",
)

### Zoom in on the soma and the AIS + slowed down

In [ ]:
# Activity 4: the same dashboard, zoomed in on the soma and AIS and slowed down

####### PARAMETERS ##########
ZOOM_UM          = 60     # µm : half-width of the 3D view
ZOOM_BEFORE_PEAK = 0.8    # ms : start playing this long before the soma's spike peak
ZOOM_AFTER_PEAK  = 0.3    # ms : and stop this long after it
FRAME_MS         = 0.005  # ms of simulated time per animation frame
FRAME_DELAY      = 100    # ms of real time per frame (larger = slower playback)
COLOUR_GAMMA     = 3      # 1 = linear colour scale; larger = more of the colours near the AP peak
#############################

soma_peaks = find_spikes(v['soma'])
if len(soma_peaks) == 0:
    print("No spike at the soma: set PULSE_AMP above threshold and re-run the cells above.")
else:
    t_soma_peak = t[soma_peaks[0]]

    view_centre = seg_xyz(h.initseg(0.7))

    # Traces and markers: the soma, the point where the spike starts, and the
    # furthest point of the axon inside the 3D view
    origin_seg, origin_v = spike_origin(t, v_segs)
    edge_seg, edge_v = furthest_in_view(v_segs, view_centre, ZOOM_UM)
    zoom_sites = [('Soma', h.soma(0.5), v['soma'], '#000000'),
                  (f'Spike starts ({h.distance(h.soma(0.5), origin_seg):.0f} µm)', origin_seg, origin_v, '#e67e22'),
                  (f'Axon ({h.distance(h.soma(0.5), edge_seg):.0f} µm)', edge_seg, edge_v, '#1baf7a')]
    print(f"The spike starts in {origin_seg.sec.name()}, {h.distance(h.soma(0.5), origin_seg):.0f} µm from the "
          f"centre of the soma (the AIS runs from {h.distance(h.soma(0.5), h.initseg(0)):.0f} to "
          f"{h.distance(h.soma(0.5), h.initseg(1)):.0f} µm).")

    zoom_dashboard = plot_interactive_dashboard(
        t, v_all,
        sites=zoom_sites,
        panels=[('Vm', [s[0] for s in zoom_sites])],
        seg_recorders=v_segs,              # colour the soma, AIS and thin segment segment by segment
        view_um=ZOOM_UM,
        view_centre=view_centre,
        mark_soma=True,
        t_window=(t_soma_peak - ZOOM_BEFORE_PEAK, t_soma_peak + ZOOM_AFTER_PEAK),
        frame_ms=FRAME_MS,
        frame_delay=FRAME_DELAY,
        colour_gamma=COLOUR_GAMMA,
        protocol=[('IClamp', PULSE_DELAY, PULSE_DELAY + PULSE_DUR, 'orange')],
        panel_height=380,
        title=f"Soma and AIS, played {FRAME_DELAY / FRAME_MS:,.0f}× slower than real time",
    )

### How fast does the action potential travel along the axon?
The time at which the spike peaks at each site is plotted against the site's distance from the soma. Along the axon the points fall on a straight line; its slope is 1 / conduction velocity.

**Questions:** which site fires first? How fast does the action potential propagate along the axon (in m/s)? How long would it take to travel the ~4 cm from the eye to the brain at this speed?

In [ ]:
# Activity 4: conduction velocity along the axon
sites = ([('Soma', v['soma'], 0.0), ('AIS', v['initseg'], h.distance(h.soma(0.5), h.initseg(0.5)))]
         + [(label, trace, axon_dist[label]) for label, trace in va.items()])
times = []
for label, trace, d in sites:
    pk = find_spikes(trace)
    pk = pk[t[pk] >= PULSE_DELAY]
    times.append(t[pk[0]] if len(pk) else np.nan)
times = np.array(times)

fig, ax = make_figure(figsize=(7, 4.5))
for (label, trace, d), tp in zip(sites, times):
    if np.isfinite(tp):
        ax.plot(d / 1000, tp - PULSE_DELAY, 'o', ms=9, color=SITE_COLOURS[label], mec='k', mew=0.5, label=label)
axon_pts = [(d, tp) for (label, trace, d), tp in zip(sites, times) if label.startswith('Axon') and np.isfinite(tp)]
cv = None
if len(axon_pts) >= 2:
    d_ax, t_ax = np.array(axon_pts).T
    slope, icpt = np.polyfit(d_ax, t_ax - PULSE_DELAY, 1)        # ms per µm
    cv = 1 / slope / 1000                                     # µm/ms -> m/s
    xs = np.array([0, d_ax.max()])
    ax.plot(xs / 1000, slope * xs + icpt, '--', color='grey', lw=1, label=f'fit: {cv:.2f} m/s')
ax.set_xlabel('Distance from soma (mm)')
ax.set_ylabel('Time of spike peak after pulse onset (ms)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for (label, trace, d), tp in zip(sites, times):
    print(f"{label:>14s}: {d:6.0f} µm, " + ("no spike" if np.isnan(tp) else f"peaks {tp - PULSE_DELAY:6.2f} ms after the pulse starts"))
if np.all(np.isnan(times)):
    print("\nNo action potential: set PULSE_AMP and PULSE_DUR above threshold and re-run Activity 4.")
else:
    print(f"\nFirst site to peak: {sites[int(np.nanargmin(times))][0]}")
if cv:
    print(f"Conduction velocity along the axon: {cv:.2f} m/s  ->  ~4 cm (eye to brain) would take {40 / cv:.0f} ms")

## Activity 5: Axonal conduction and axon diameter

In [ ]:
# Activity 5: conduction velocity for one axon diameter

####### PARAMETERS ##########
AXON_DIAM   = 1.0    # µm (the model's axon is 1 µm)
PULSE_AMP   = 0      # pA (Fill this in yourself! Use a value comfortably above threshold)
PULSE_DUR   = 0      # ms (Fill this in yourself!)
PULSE_DELAY = 5      # ms
T_STOP      = 30     # ms
DT          = 0.025  # ms : time step
#############################

# Change the axon, and set the pulse (reusing Activity 4's clamp and recordings)
axon_diam_default = 1.0                   # µm : the model's axon
h.axon.diam = AXON_DIAM
mc_iclamp.delay = PULSE_DELAY
mc_iclamp.dur   = PULSE_DUR
mc_iclamp.amp   = PULSE_AMP / 1000        # nA

# Run (as in Activity 4)
h.dt      = DT
h.tstop   = T_STOP
h.celsius = 33
h.finitialize(-62.5)
for sec in h.allsec():
    if h.ismembrane('ca_ion', sec=sec):
        sec.cai = 0.0001
h.fcurrent()
h.frecord_init()
h.continuerun(T_STOP)
h.axon.diam = axon_diam_default           # restore the model

t  = np.array(t_vec)
v  = {name: np.array(vec) for name, vec in v_rec.items()}
va = {label: np.array(vec) for label, vec in v_axon.items()}

plot_hh_panels(t, np.array(i_vec) * 1000,
               [('Soma', v['soma'], SITE_COLOURS['Soma']), ('AIS', v['initseg'], SITE_COLOURS['AIS'])],
               m_soma, h_soma, n_soma, h.soma(0.5).gnabar_spike, h.soma(0.5).gkbar_spike,
               extra_panels=[('Axon Vm (mV)', [('Soma', v['soma'], '#bbbbbb')]
                              + [(l, tr, SITE_COLOURS[l]) for l, tr in va.items()])],
               title=f"Axon diameter {AXON_DIAM} µm: {PULSE_AMP} pA for {PULSE_DUR} ms into the soma")

# Conduction velocity between the first two axon sites (0.7 mm and 2.8 mm)
(lab1, seg1, d1), (lab2, seg2, d2) = axon_sites[0], axon_sites[1]
p1, p2 = find_spikes(va[lab1]), find_spikes(va[lab2])
if len(p1) and len(p2):
    cv = (d2 - d1) / (t[p2[0]] - t[p1[0]]) / 1000           # µm/ms -> m/s
    print(f"Axon diameter {AXON_DIAM} µm: conduction velocity {cv:.2f} m/s")
    if AXON_DIAM != 1.0 and 'CV_1UM' in globals():
        print(f"  √diameter prediction from the 1 µm axon: {CV_1UM * np.sqrt(AXON_DIAM):.2f} m/s")
    if AXON_DIAM == 1.0:
        CV_1UM = cv                                         # kept for the √diameter prediction
else:
    print(f"Axon diameter {AXON_DIAM} µm: the spike did not reach 2.8 mm along the axon")